In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchinfo import summary
from lib.trainer import Trainer
import os

In [ ]:
# define hyperparameters
batch_size = 32
learning_rate = 1e-2
num_classes = 10
num_epochs = 10

In [ ]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.mps.is_available() else "cpu"
)

In [ ]:
DATA_PATH = os.path.join(os.path.dirname(os.path.abspath("./")), "data")

In [ ]:
train_dataset = torchvision.datasets.MNIST(
    root=DATA_PATH,
    train=True,
    download=True,
    transform=transforms.Compose(
        [
            transforms.Resize((32, 32)),
            transforms.ToTensor(),
            transforms.Normalize(mean=0.07843, std=0.7843),
        ]
    ),
)

In [ ]:
val_dataset = torchvision.datasets.MNIST(
    root=DATA_PATH,
    train=False,
    download=True,
    transform=transforms.Compose(
        [
            transforms.Resize((32, 32)),
            transforms.ToTensor(),
            transforms.Normalize(mean=0.07843, std=0.7843),
        ]
    ),
)

In [ ]:
# define dataloaders
train_dataloader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True
)
val_dataloader = torch.utils.data.DataLoader(
    val_dataset, batch_size=batch_size, shuffle=True
)

In [ ]:
# define lenet
class LeNet5(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        C1 = nn.Sequential(nn.Conv2d(1, 6, kernel_size=(5, 5)), nn.Tanh())
        S2 = nn.Sequential(
            nn.AvgPool2d(kernel_size=(2, 2), stride=2),
            nn.Tanh(),
        )
        C3 = nn.Sequential(
            nn.Conv2d(6, 16, kernel_size=(5, 5)),
            nn.Tanh(),
        )
        S4 = nn.Sequential(
            nn.AvgPool2d(kernel_size=(2, 2), stride=2),
            nn.Tanh(),
        )
        C5 = nn.Sequential(
            nn.Conv2d(16, 120, kernel_size=(5, 5)),
            nn.Tanh(),
        )
        F6 = nn.Sequential(nn.Linear(120, 84), nn.Tanh())
        output_layer = nn.Linear(84, 10)

        self.layers = nn.Sequential(C1, S2, C3, S4, C5, nn.Flatten(), F6, output_layer)

    def forward(self, x):
        output = self.layers(x)
        return output


summary(LeNet5(num_classes))

In [ ]:
model = LeNet5().to(device)
loss_criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, learning_rate, 0.9)

In [ ]:
trainer = Trainer(
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    model=model,
    loss_fn=loss_criterion,
    optimizer=optimizer,
    scheduler=scheduler,
)
trainer.fit(epochs=10)

In [ ]:
trainer.plot()